# CellBridge

CellBridge predicts a donor's protein response from the change in cell-level RNA and a few anchor antibodies. The fit is closed form. This notebook uses synthetic donors so it does not touch a sealed cohort.

An AnnData object with the same layout works through `fit_anndata`: `X` is the model matrix, `obs` has `donor` and `arm`, and `obsm['protein']` is the protein matrix. The repository is under the MIT license in `LICENSE`.

In [ ]:
import numpy as np
from cellbridge import fit

rng = np.random.default_rng(0)
beta = np.array([1.0, 0.0, 0.0, 0.0, 0.2, 0.0])
donors = []
for i in range(4):
    zc = rng.normal(size=(24, 6))
    zp = rng.normal(size=(24, 6))
    donors.append({
        "donor": f"d{i}",
        "z_control": zc,
        "y_control": (zc @ beta)[:, None],
        "z_perturbed": zp,
        "y_perturbed": (zp @ beta + 0.4)[:, None],
    })
model = fit(donors, n_genes=4, n_anchors=0, ks=(1,), mixes=((0.0, 0.0),), lambdas=(1e-3,), strategy="shared", top=1.0)
point = model.predict(np.zeros(6))
parts = model.contributions(np.zeros(6), 0)
low, high = model.intervals(np.zeros(6), np.full(20, 0.2), alpha=0.1)
print(float(point[0, 0]), float(low[0, 0]), float(high[0, 0]), float(parts.sum()))
